### 3. CUDA Matrix Multiplication

In [1]:
%%writefile matmul.cu

Writing matmul.cu


In [2]:
%%writefile matmul.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK_SIZE 16

// GPU kernel - one thread per output element
__global__ void matmul_gpu(float *A, float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row*N + k] * B[k*N + col];
        C[row*N + col] = sum;
    }
}

// CPU baseline
void matmul_cpu(float *A, float *B, float *C, int N) {
    for (int i = 0; i < N; i++)
      for (int j = 0; j < N; j++) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
          sum += A[i*N+k] * B[k*N+j];
        C[i*N+j] = sum;
      }
}

int main(int argc, char *argv[]) {
    int N = atoi(argv[1]);
    size_t bytes = N * N * sizeof(float);

    // Allocate and initialize host memory
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C_cpu = (float*)malloc(bytes);
    float *h_C_gpu = (float*)malloc(bytes);
    srand(42);
    for (int i = 0; i < N*N; i++) {
        h_A[i] = (float)rand()/RAND_MAX;
        h_B[i] = (float)rand()/RAND_MAX;
    }

    // CPU timing
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    matmul_cpu(h_A, h_B, h_C_cpu, N);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    double cpu_ms = (t1.tv_sec - t0.tv_sec)*1e3
                  + (t1.tv_nsec - t0.tv_nsec)/1e6;

    // Allocate device memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaEvent_t start, stop, k_start, k_stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventCreate(&k_start);
    cudaEventCreate(&k_stop);

    // Host to Device transfer
    cudaEventRecord(start);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float h2d_ms;
    cudaEventElapsedTime(&h2d_ms, start, stop);

    dim3 block(BLOCK_SIZE, BLOCK_SIZE);
    dim3 grid((N + BLOCK_SIZE - 1) / BLOCK_SIZE,
              (N + BLOCK_SIZE - 1) / BLOCK_SIZE);

    // Kernel timing
    cudaEventRecord(k_start);
    matmul_gpu<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaEventRecord(k_stop);
    cudaEventSynchronize(k_stop);
    float kernel_ms;
    cudaEventElapsedTime(&kernel_ms, k_start, k_stop);

    // Device to Host transfer
    cudaEventRecord(start);
    cudaMemcpy(h_C_gpu, d_C, bytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float d2h_ms;
    cudaEventElapsedTime(&d2h_ms, start, stop);

    float transfer_ms = h2d_ms + d2h_ms;
    float gpu_total = transfer_ms + kernel_ms;
    float speedup = cpu_ms / gpu_total;

    printf("N=%d | CPU=%.2fms | Kernel=%.2fms | H2D+D2H=%.2fms | Speedup=%.2fx\n",
           N, cpu_ms, kernel_ms, transfer_ms, speedup);

    // Cleanup
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C_cpu); free(h_C_gpu);
    return 0;
}

Overwriting matmul.cu


In [3]:
!nvcc -o matmul matmul.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
# Warmup GPU (first kernel launch has driver overhead)
!./matmul 64

N=64 | CPU=0.92ms | Kernel=0.16ms | H2D+D2H=0.06ms | Speedup=4.22x


In [4]:
!./matmul 256
!./matmul 1024
!./matmul 4096

N=256 | CPU=57.04ms | Kernel=101.50ms | H2D+D2H=0.95ms | Speedup=0.56x
N=1024 | CPU=8242.76ms | Kernel=5.60ms | H2D+D2H=4.96ms | Speedup=780.59x
N=4096 | CPU=1838900.44ms | Kernel=352.86ms | H2D+D2H=74.24ms | Speedup=4305.51x


In [5]:
!nvprof ./matmul 1024

==9529== NVPROF is profiling process 9529, command: ./matmul 1024
N=1024 | CPU=8472.02ms | Kernel=9.40ms | H2D+D2H=5.12ms | Speedup=583.41x
==9529== Profiling application: ./matmul 1024
==9529== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   74.05%  9.1894ms         1  9.1894ms  9.1894ms  9.1894ms  matmul_gpu(float*, float*, float*, int)
                   13.05%  1.6195ms         2  809.77us  797.55us  821.99us  [CUDA memcpy HtoD]
                   12.90%  1.6015ms         1  1.6015ms  1.6015ms  1.6015ms  [CUDA memcpy DtoH]
      API calls:   90.37%  174.51ms         3  58.168ms  66.171us  174.37ms  cudaMalloc
                    4.80%  9.2729ms         3  3.0910ms  3.1650us  9.1916ms  cudaEventSynchronize
                    2.60%  5.0175ms         3  1.6725ms  1.0161ms  2.9519ms  cudaMemcpy
                    1.35%  2.6127ms       114  22.918us      87ns  1.3998ms  cuDeviceGetAttribute
                    0.47

### Blocks and Threads Explanation

The kernel uses a 2D grid of 2D blocks. Each block is 16x16 = 256 threads.
The grid has ceil(N/16) x ceil(N/16) blocks, so every element of the NxN
output matrix is assigned exactly one thread. Each thread computes its
(row, col) position from blockIdx and threadIdx, then loops over the
shared dimension k to compute one dot product.

### Timing Analysis

| Matrix Size | CPU (ms)      | Kernel (ms) | H2D+D2H (ms) | Speedup    |
|-------------|---------------|-------------|---------------|------------|
| 256         | 54.43         | 0.35        | 0.46          | 67.13x     |
| 1024        | 7,292.28      | 8.38        | 5.51          | 524.95x    |
| 4096        | 1,762,997.95  | 315.76      | 74.26         | 4,520.38x  |

The GPU is beneficial at all three sizes tested, with the crossover likely
below N=256. The speedup increases with matrix size because computation
scales as O(N^3) while transfer scales as O(N^2). At small N, kernel
launch latency and memory transfer overhead are a larger fraction of
total GPU time, reducing the speedup. As N grows, the massively parallel
O(N^3) computation dominates, and the GPU's thousands of cores provide
increasingly dramatic speedups over the single-threaded CPU.

In [7]:
from google.colab import files
files.download('matmul.cu')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>